# Глава 5 — Tools

Реестр функций, описания и схемы, выбор инструмента моделью, выполнение и запись observation. Сравниваем JSON в тексте (`Tools`) и native function calling (`NativeTools`).

Как в главе 5, один `run()` делает один шаг: возвращает ответ модели или результат инструмента. Автономный цикл — глава 6.

Запустите Jupyter из корня проекта с зависимостями из README. Здесь используется установленная `gemma4:e4b`; текстовый вариант работает с `think=False`. В книге для текстового варианта используется `gemma3:12b` — можно выбрать её, если она уже скачана.

In [ ]:
from agent import TinyAgent
from llm import LLM, Response
from memory import Memory
from toolbox import multiply
from tools import Tools, NativeTools, tool_to_schema
from illustrated_agents.utils import TrajectoryViewer

llm = LLM(model="gemma4:e4b", think=False, temperature=0)

## Создание, регистрация и описание инструмента

Модель выбирает имя и аргументы. Код вызывает только функцию, которая явно зарегистрирована под этим именем.

In [ ]:
print(multiply("3.1", "6.5"))
tools = Tools()
tools.add_tool("multiply", multiply, "Multiplies two numbers: multiply(a: str, b: str)")
print(tools.prompt)

## Разбор и выполнение без модели

Этот пример позволяет отдельно проверить инструмент, прежде чем проверять выбор инструмента LLM.

In [ ]:
response = Response(content='{"tool": "multiply", "kwargs": {"a": "3.1", "b": "6.5"}}')
parsed = tools.parse(response)
print(parsed)
print(tools.execute(parsed))

## Вызов инструмента через JSON в тексте

Результат записывается как user-сообщение `OBSERVATION: ...`. Память отличает его от нового запроса пользователя.

In [ ]:
agent = TinyAgent(llm=llm, memory=Memory(), tools=tools)
print(agent.run("Use the multiply tool to calculate 5.1 times 7.3."))
print(agent.memory.get_messages())
print(agent.trajectory.runs)
assert agent.trajectory.runs[0]["steps"][0].action["tool"] == "multiply"

In [ ]:
TrajectoryViewer(agent.trajectory)

In [ ]:
print(agent.run("State the previous tool result without calling any tools. Be concise."))

## Подтверждение перед выполнением

Для инструментов из `requires_approval` стандартный режим спрашивает в терминале имя и аргументы (`[y/N]`). В notebook используем явный callback, который отказывает, чтобы запуск всех ячеек не ожидал ввода. `multiply` сам по себе не изменяет внешний мир и здесь служит демонстрацией.

In [ ]:
approval_requests = []

def deny(name, kwargs):
    approval_requests.append((name, kwargs))
    return False

guarded = Tools(requires_approval=["multiply"], approval=deny)
guarded.add_tool("multiply", multiply)
print(guarded.execute(parsed))
print("Approval requested for:", approval_requests)

## Неизвестные инструменты и ошибки

Ошибки исполнения превращаются в observations. Некорректный JSON или несколько вызовов за один шаг отклоняются до исполнения.

In [ ]:
print(tools.execute(Response(tool_call={"tool": "unknown", "kwargs": {}})))
print(tools.execute(Response(tool_call={"tool": "multiply", "kwargs": {"a": "not a number", "b": "3"}})))

## Native function calling

Схема строится по Python-сигнатуре; регистрационное имя и описание сохраняются. LLM возвращает структурированный вызов. В памяти хранится его исходная структура и tool-результат с соответствующим `tool_call_id`.

In [ ]:
print(tool_to_schema(multiply))
native_tools = NativeTools()
native_tools.add_tool("multiply", multiply)
print(native_tools.schemas)

In [ ]:
native_agent = TinyAgent(llm=llm, memory=Memory(), tools=native_tools)
print(native_agent.run("Use the multiply tool to calculate 5.1 times 7.3."))
print(native_agent.memory.get_messages())
print(native_agent.trajectory.runs)
assert native_agent.trajectory.runs[0]["steps"][0].action["tool"] == "multiply"

In [ ]:
TrajectoryViewer(native_agent.trajectory)

In [ ]:
print(native_agent.run("State the previous tool result without calling any tools. Be concise."))

## Что пока остаётся за рамками

MCP, Skills и обучение моделей вызову инструментов рассмотрены в главе теоретически. Этот notebook не подключает внешние сервисы и не обучает модель. Native-клиент запрашивает один вызов за шаг и явно отклоняет несколько вызовов, если backend всё же их возвращает. В главе 6 появится автономный цикл.